# Notebook 1 — The Simulator Environment

**Goal.** By the end of this notebook you will understand *what the robot sees* (the observation) and *what it can do* (the action), and you will have built a small `GraspEnv` wrapper that every later notebook reuses.

This is the **first box** of the project pipeline:

```
[Simulator]  ->  [Expert]  ->  [Database]  ->  [Training]  ->  [Evaluation]
   ^ you are here
```

We use **panda-gym**: a ready-made simulation of a Franka Panda robot arm doing pick-and-place, built on the PyBullet physics engine. It runs on your Mac laptop, CPU only — no GPU, no lab, no real robot.

> How to use this notebook: run each cell top to bottom, read the short note above it, and look at what it prints. Boxes marked **Your turn** are tiny experiments for you.

## 0. Setup (do this once)

In a terminal, inside the project folder:

```bash
uv venv --python 3.11
source .venv/bin/activate
uv pip install panda-gym torch numpy pandas matplotlib jupyter
```

Then launch with `jupyter lab`, or open this file in VS Code / Cursor and pick the `.venv` kernel.

**Two rules that trip up everyone on the first run:**
1. You must `import panda_gym` *before* `gym.make(...)`. Importing panda-gym is what *registers* the `PandaPickAndPlace-v3` environment id. Without it you get a `NameNotFound` error. This is not an ordinary import — it is a required side effect.
2. On an Apple-Silicon Mac, the pop-up 3D window (`render_mode='human'`) is unreliable and often freezes. So we run **headless** (no window) for everything here. There is an optional window cell near the end; nothing depends on it.

In [4]:
!pip install gymnasium
!pip install panda_gym


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 1.9 MB/s  0:00:49m0:00:0100:02m
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pybullet (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [8488 lines of output]
      macosx-11.1-arm64
      numpy is disabled. getCameraImage maybe slower.
      darwin!
      found resource files: 3568
      -- checker_grid.jpg
      -- duckCM.png
      -- textured_sphere_smooth.obj
      -- plane_implicit.urdf
      -- TwoJointRobot_w_fixedJoints.urdf
      -- cube.png
      -- cartpole.urdf
      -- TwoJointRobot_wo_fixedJoints.urdf
      -- cloth_z_up.obj
      -- plane_transparent.urdf
      -- pr2_gripper.urdf
      -- stadium.sdf
      -- checker_blue.png
      -- c

In [5]:
# Setup check — run this first.
import numpy as np
import gymnasium as gym
import panda_gym  # REQUIRED: this line registers the Panda environments

print('gymnasium version:', gym.__version__)
print('panda_gym version:', panda_gym.__version__)
print('numpy version    :', np.__version__)
print()
print('If this cell ran without an error, your setup works.')

ModuleNotFoundError: No module named 'panda_gym'

## 1. Make the environment (headless)

`gym.make(...)` builds the physics world. We pass no `render_mode`, which means **no window** — the fast mode we use for collecting data and training.

In [ ]:
env = gym.make('PandaPickAndPlace-v3')   # render_mode omitted => headless (no window)
print(env)

## 2. The two spaces

Every gym environment declares:
- an **observation space** — the shape of what the robot senses, and
- an **action space** — the shape of what the robot can command.

For pick-and-place the observation is a **dictionary** (three named parts) and the action is a **box of 4 numbers**, each between -1 and 1.

In [ ]:
print('OBSERVATION space:')
print(env.observation_space)
print()
print('ACTION space:')
print(env.action_space)

## 3. Reset, and look at one observation

`env.reset()` starts a fresh episode (the object and goal get new random positions) and returns the first observation. We pass `seed=0` so it is reproducible.

The observation is a dict with three keys:
- `observation` — the main sensor vector (19 numbers),
- `achieved_goal` — where the object currently is (3 numbers),
- `desired_goal` — where the object should end up (3 numbers).

In [ ]:
obs, info = env.reset(seed=0)

print('obs is a', type(obs).__name__, 'with keys:', list(obs.keys()))
print()
for key, value in obs.items():
    print(f'{key:14s} -> {len(value):2d} numbers')
print()
print('info:', info)

## 4. What does each observation number mean?

The 19 numbers in `obs['observation']` are, in order:

| index | meaning |
|------:|---------|
| 0..2  | gripper (end-effector) position x, y, z |
| 3..5  | gripper velocity vx, vy, vz |
| 6     | fingers width (how open the gripper is) |
| 7..9  | **object** position x, y, z |
| 10..12| object rotation (roll, pitch, yaw) |
| 13..15| object velocity |
| 16..18| object angular velocity |

**Important detail you will rely on later:** the object position sits at indices **7, 8, 9**. That is only true because `fingers_width` takes up index 6. (In the simpler no-gripper tasks the object would start at index 6 instead.) The scripted expert in the next notebook uses `obs['observation'][7:10]` to find the object — now you know why.

In [ ]:
OBS_LABELS = [
    'ee_x  (gripper x)', 'ee_y  (gripper y)', 'ee_z  (gripper z)',
    'ee_vx (gripper vel x)', 'ee_vy (gripper vel y)', 'ee_vz (gripper vel z)',
    'fingers_width',
    'obj_x (object x)', 'obj_y (object y)', 'obj_z (object z)',
    'obj_roll', 'obj_pitch', 'obj_yaw',
    'obj_vx', 'obj_vy', 'obj_vz',
    'obj_wx', 'obj_wy', 'obj_wz',
]

main = obs['observation']
print(f'obs[observation] has {len(main)} numbers:')
for i, (label, value) in enumerate(zip(OBS_LABELS, main)):
    print(f'  [{i:2d}] {label:22s} = {value: .3f}')

print()
print('achieved_goal (object position now):', np.round(obs['achieved_goal'], 3))
print('desired_goal  (where it should go) :', np.round(obs['desired_goal'], 3))

## 5. What can the robot do? (the action)

An action is **4 numbers**, each clamped to [-1, 1]:

| index | meaning |
|------:|---------|
| 0 | move gripper in x |
| 1 | move gripper in y |
| 2 | move gripper in z |
| 3 | gripper: positive = open, negative = close |

> Note for later (the expert notebook): the first three numbers are a *normalized* step, scaled internally by about 0.05 m, and number 3 is *added* to the current gripper opening. You do not need this yet — just remember the four slots.

In [ ]:
action = env.action_space.sample()   # one random action
names = ['move_x', 'move_y', 'move_z', 'gripper']
print('a random action:')
for name, value in zip(names, action):
    print(f'  {name:8s} = {value: .3f}')

## 6. Watch random actions (and meet the reward)

Let us step the environment with **random** actions and see what comes back. Each `env.step(action)` returns five things:

`obs, reward, terminated, truncated, info = env.step(action)`

- `reward` is **sparse**: `0.0` on a step where the object is at the goal, `-1.0` otherwise.
- `info['is_success']` is `True` / `False` — the honest success flag we measure later.
- `terminated` becomes `True` when the task is solved; `truncated` becomes `True` when time runs out (episodes are capped at 50 steps).

Random actions almost never succeed — that is the whole point. Replacing random with *smart* is the rest of the project.

In [ ]:
obs, info = env.reset(seed=0)
successes = 0

for step in range(50):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    if info['is_success']:
        successes += 1
    if terminated or truncated:
        obs, info = env.reset()

print(f'ran 50 random steps; success flags seen: {successes}')
print('last reward:', reward)
print('(random almost never solves it — expected)')

## 7. Flatten the observation into one vector

The neural network (later) wants **one flat list of numbers**, not a dictionary. So we join the parts we care about into a single vector.

We keep `observation` (19) + `desired_goal` (3) = **22 numbers**. We drop `achieved_goal` because it is just a copy of the object position, which already lives inside `observation`.

**The one rule that matters:** the order must be *exactly the same* every time — at data-collection time and at evaluation time. If the order ever changes, the network sees scrambled inputs. This fixed order is a *contract*.

In [ ]:
def flatten_obs(obs_dict):
    parts = [obs_dict['observation'], obs_dict['desired_goal']]
    return np.concatenate(parts).astype(np.float32)

flat = flatten_obs(obs)
print('flat observation length:', len(flat))
print('dtype                  :', flat.dtype)
print(flat)

## 8. Pin down the sizes

We fix two numbers the whole project agrees on:
- `OBS_DIM = 22` — the length of the flat observation (the network **input** size).
- `ACTION_DIM = 4` — the length of an action (the network **output** size).

Here is the full 22-number map, including the goal at the end:

| flat index | meaning |
|-----------:|---------|
| 0..2   | gripper position |
| 3..5   | gripper velocity |
| 6      | fingers width |
| 7..9   | object position |
| 10..12 | object rotation |
| 13..15 | object velocity |
| 16..18 | object angular velocity |
| 19..21 | **desired goal** x, y, z |

In [ ]:
OBS_DIM = 22
ACTION_DIM = 4

FLAT_LABELS = OBS_LABELS + ['goal_x', 'goal_y', 'goal_z']
assert len(FLAT_LABELS) == OBS_DIM

print(f'the {OBS_DIM} numbers the network will see:')
for i, (label, value) in enumerate(zip(FLAT_LABELS, flat)):
    print(f'  [{i:2d}] {label:22s} = {value: .3f}')

## 9. Wrap it all up: the `GraspEnv` class

Now we package everything into one small class. This is the **deliverable of this notebook** — the clean box every later step (expert, training, evaluation) imports, instead of touching panda-gym directly.

`GraspEnv` does four simple things:
- `reset(seed)` — start an episode and return the **flat** observation.
- `step(action)` — advance one step and return the **flat** observation (plus reward / done flags / info).
- keeps the **raw dict** around (`self.raw_obs`) because the scripted expert next week needs `obs['observation']` and `obs['desired_goal']` directly.
- `close()` — shut the simulator down.

Notice it is the *only* place that calls `gym.make`. Everything downstream depends on this contract, not on panda-gym.

In [ ]:
class GraspEnv:
    def __init__(self, render_mode=None):
        # render_mode=None -> headless (fast). Use 'human' only to watch (see section 12).
        self._env = gym.make('PandaPickAndPlace-v3', render_mode=render_mode)
        self.obs_dim = OBS_DIM
        self.action_dim = ACTION_DIM
        self.action_space = self._env.action_space
        self.raw_obs = None

    def reset(self, seed=None):
        obs, info = self._env.reset(seed=seed)
        self.raw_obs = obs                      # keep the dict for the expert
        return flatten_obs(obs), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self._env.step(action)
        self.raw_obs = obs
        return flatten_obs(obs), reward, terminated, truncated, info

    def close(self):
        self._env.close()

In [ ]:
# quick check that the wrapper behaves
genv = GraspEnv()
flat0, info0 = genv.reset(seed=0)
print('reset -> flat obs shape:', flat0.shape, 'dtype:', flat0.dtype)

flat1, reward, terminated, truncated, info = genv.step(genv.action_space.sample())
print('step  -> flat obs shape:', flat1.shape)
print('reward:', reward, '| terminated:', terminated, '| truncated:', truncated)
print('raw dict still available:', list(genv.raw_obs.keys()))

## 10. Run one full episode

A *policy* is just a function that takes an observation and returns an action. Here the policy is `random_policy`. Later it becomes the scripted expert, then the trained network — but the loop below never changes. Writing this loop once is why the wrapper is worth it.

In [ ]:
def random_policy(flat_obs):
    return genv.action_space.sample()

def run_episode(env, policy, seed=None):
    flat_obs, info = env.reset(seed=seed)
    total_reward = 0.0
    steps = 0
    while True:
        action = policy(flat_obs)
        flat_obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        steps += 1
        if terminated or truncated:
            break
    return {'success': bool(info['is_success']), 'steps': steps, 'total_reward': total_reward}

result = run_episode(genv, random_policy, seed=0)
print('episode result:', result)

## 11. Seeding: why experiments are reproducible

If you reset with the *same* seed, the object and goal appear in the *same* place, so the first observation is identical. Fixed seeds are what make later experiments honest — a lucky run cannot masquerade as a good method.

In [ ]:
a, _ = genv.reset(seed=0)
b, _ = genv.reset(seed=0)
print('same seed -> identical first observation:', np.allclose(a, b))

c, _ = genv.reset(seed=1)
print('different seed -> different observation  :', not np.allclose(a, c))

## 12. Headless vs. window vs. video

Three render modes:
- `render_mode=None` — **no window, fast**. Use this for collecting data and training (everything above).
- `render_mode='human'` — a 3D window to *watch* the arm. **On Apple-Silicon Macs this often freezes**, so treat it as optional.
- `render_mode='rgb_array'` — returns each frame as an image; used to record an `.mp4` for the final presentation.

The optional cell below opens a window. If it hangs, interrupt the kernel (the stop button) — nothing else in the notebook needs it.

In [ ]:
# OPTIONAL — a short window. On Mac it may freeze; skip if so.
# Remove the # on the lines below to try it.
#
# watch_env = GraspEnv(render_mode='human')
# watch_env.reset(seed=0)
# for _ in range(60):
#     watch_env.step(watch_env.action_space.sample())
# watch_env.close()
print('GUI cell is commented out on purpose — enable it only if you want to watch.')

## 13. Save your work into the package

Right now `flatten_obs`, the size constants, and `GraspEnv` live only in this notebook. Move them into the `ilgrasp` package so scripts can reuse them:

- **`src/ilgrasp/sim/observation.py`** — put `OBS_DIM`, `ACTION_DIM`, `OBS_LABELS`, `FLAT_LABELS`, and `flatten_obs` here.
- **`src/ilgrasp/sim/env.py`** — put the `GraspEnv` class here (it imports `flatten_obs`, `OBS_DIM`, `ACTION_DIM` from `observation`).
- **`src/ilgrasp/sim/__init__.py`** — expose the public API:

```python
from .observation import flatten_obs, OBS_DIM, ACTION_DIM
from .env import GraspEnv

__all__ = ['GraspEnv', 'flatten_obs', 'OBS_DIM', 'ACTION_DIM']
```

After that, any script can simply do `from ilgrasp.sim import GraspEnv`.

> To make `import ilgrasp` work at all, the project needs a `pyproject.toml` and an editable install (`uv pip install -e .`). Pair with your mentor on this once — it is a 10-minute, one-time setup.

## 14. What you built, and what comes next

**You now have:**
- a clear picture of the 22-number observation and the 4-number action,
- a `flatten_obs` function with a fixed ordering contract,
- a `GraspEnv` wrapper with `reset` / `step` / `close`,
- a reusable `run_episode` loop and a working seed check.

**Next notebook — the scripted expert.** You will replace `random_policy` with a hand-written controller that actually solves the task (approach the object, close the gripper, lift, carry to the goal). It reads `env.raw_obs['observation'][7:10]` (the object position you found above) and `env.raw_obs['desired_goal']`.

> One thing that will bite you there: the action is *normalized*. To move the gripper by a real distance, multiply the direction by roughly 20 and clip to [-1, 1]; the gripper slot is *added* to the current opening (positive opens, negative closes). We will handle it together.

---

### Your turn (small experiments)
1. In the labeled-observation cell (section 4), find where the object starts. Reset with a few different seeds and watch `obj_x, obj_y, obj_z` change.
2. Send the action `[0, 0, -1, 0]` (push straight down) for 10 steps and print the gripper z (`obs['observation'][2]`) each step. Does it go down?
3. Send `[0, 0, 0, -1]` (close gripper) repeatedly and watch `fingers_width` (`obs['observation'][6]`) shrink.

In [ ]:
# tidy up: close the environments when you are done
env.close()
genv.close()
print('environments closed.')